# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_10 — Realized Volatility Estimators from OHLC Data

### Research question

How do different OHLC volatility estimators behave when returns include intraday diffusion, drift, overnight gaps, and jumps?

This notebook compares four classical volatility estimators:

1. **Close-to-close**
2. **Parkinson**
3. **Garman-Klass**
4. **Rogers-Satchell**

The notebook is designed as the bridge into later volatility forecasting notebooks, especially:

```text
03_02_har_realized_volatility_forecasting
```

It focuses on:

- estimator formulas;
- assumptions;
- bias and error under different market regimes;
- annualised realised-volatility features;
- volatility clustering diagnostics;
- HAR-RV-ready feature construction;
- why the estimator choice matters before fitting GARCH, HAR-RV, or ML volatility models.

The central warning is:

> Volatility is not one object. Close-to-close, intraday range-based, jump-inclusive, and overnight-inclusive estimators answer related but different questions.

## 1. Why realised volatility estimation matters

Volatility enters almost every quantitative finance workflow:

- risk targeting;
- position sizing;
- VaR and expected shortfall;
- option pricing;
- delta hedging;
- trend-following risk control;
- volatility forecasting;
- regime detection;
- portfolio optimisation.

A naïve estimate uses only close-to-close returns:

$$
r_t = \log C_t - \log C_{t-1}
$$
and then computes:

$$
\hat{\sigma}^2 = \sum_t r_t^2
$$
But daily OHLC bars contain more information:

- open;
- high;
- low;
- close.

Range-based estimators use intraday high and low prices to extract additional information about the path inside each bar.

However, these estimators rely on assumptions. They can be biased by drift, jumps, overnight gaps, microstructure noise, price limits, or bad OHLC data.

## 2. Estimators covered

Let:

$$
O_t, H_t, L_t, C_t
$$
be the open, high, low, and close prices for day $t$.

Define:

$$
h_t = \log H_t,\quad
l_t = \log L_t,\quad
o_t = \log O_t,\quad
c_t = \log C_t
$$
### 2.1 Close-to-close variance

$$
\begin{aligned}
\hat{\sigma}_{CC,t}^2 &= (\log C_t - \log C_{t-1})^2
\end{aligned}
$$
This includes overnight gaps, because it compares today's close to yesterday's close.

### 2.2 Open-to-close variance

$$
\begin{aligned}
\hat{\sigma}_{OC,t}^2 &= (\log C_t - \log O_t)^2
\end{aligned}
$$
This is a useful baseline for intraday volatility.

### 2.3 Parkinson estimator

$$
\begin{aligned}
\hat{\sigma}_{P,t}^2 &= \frac{1}{4\log 2} \left(\log \frac{H_t}{L_t}\right)^2
\end{aligned}
$$
This uses only the high-low range.

### 2.4 Garman-Klass estimator

$$
\begin{aligned}
\hat{\sigma}_{GK,t}^2 &= \frac{1}{2} \left(\log \frac{H_t}{L_t}\right)^2 \\
&\quad - (2\log 2 - 1) \left(\log \frac{C_t}{O_t}\right)^2
\end{aligned}
$$
This uses open, high, low, and close.

### 2.5 Rogers-Satchell estimator

$$
\begin{aligned}
\hat{\sigma}_{RS,t}^2 &= \log \frac{H_t}{O_t} \log \frac{H_t}{C_t} \\
&\quad + \log \frac{L_t}{O_t} \log \frac{L_t}{C_t}
\end{aligned}
$$
The Rogers-Satchell estimator is designed to be robust to drift under a Brownian log-price model.

## 3. Assumption map

| Estimator | Uses | Strength | Main weakness |
|---|---|---|---|
| Close-to-close | $C_{t-1}, C_t$ | Simple, includes overnight moves | Noisy; ignores intraday range |
| Open-to-close | $O_t, C_t$ | Clean intraday baseline | Ignores high/low and overnight |
| Parkinson | $H_t, L_t$ | Efficient under zero-drift diffusion | Biased by jumps, drift, microstructure, price limits |
| Garman-Klass | $O_t,H_t,L_t,C_t$ | Very efficient under zero drift/no jumps | Sensitive to drift and opening gaps |
| Rogers-Satchell | $O_t,H_t,L_t,C_t$ | More robust to drift | Still assumes continuous intraday path and reliable OHLC |

This notebook will test these assumptions using synthetic data where the true latent variance is known.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class OHLCVolSimulationConfig:
    """
    Configuration for synthetic OHLC volatility simulation.
    """
    start: str = "2020-01-01"
    n_days: int = 1_500
    intraday_steps: int = 390
    s0: float = 100.0
    annual_vol_base: float = 0.25
    annual_drift: float = 0.08
    overnight_vol: float = 0.012
    jump_probability: float = 0.015
    jump_scale: float = 0.05
    volatility_persistence: float = 0.97
    volatility_of_volatility: float = 0.12
    seed: int = 42


config = OHLCVolSimulationConfig()
config

## 5. Synthetic OHLC data with known latent variance

We simulate a daily price path with:

1. time-varying volatility;
2. intraday Brownian diffusion;
3. daily drift;
4. overnight gaps;
5. occasional intraday jumps.

For each day, we simulate many intraday prices and then extract:

$$
O_t,\ H_t,\ L_t,\ C_t
$$
Because the simulation controls the latent volatility, we also know the approximate true intraday integrated variance for each day.

In [ ]:
def simulate_time_varying_daily_volatility(config: OHLCVolSimulationConfig) -> np.ndarray:
    """
    Simulate a persistent daily annualised volatility process.
    """
    local_rng = np.random.default_rng(config.seed)

    log_vol = np.empty(config.n_days)
    log_vol[0] = np.log(config.annual_vol_base)

    for t in range(1, config.n_days):
        log_vol[t] = (
            np.log(config.annual_vol_base) * (1 - config.volatility_persistence)
            + config.volatility_persistence * log_vol[t - 1]
            + config.volatility_of_volatility * local_rng.standard_normal()
        )

    annual_vol = np.exp(log_vol)

    # Avoid pathological synthetic values.
    annual_vol = np.clip(annual_vol, 0.05, 1.20)

    return annual_vol

In [ ]:
def simulate_ohlc_panel(config: OHLCVolSimulationConfig) -> pd.DataFrame:
    """
    Simulate OHLC data and latent variance from an intraday price process.
    """
    local_rng = np.random.default_rng(config.seed + 1)

    dates = pd.bdate_range(config.start, periods=config.n_days, tz="UTC")
    annual_vol = simulate_time_varying_daily_volatility(config)

    rows = []
    previous_close = config.s0

    for day_idx, date in enumerate(dates):
        sigma_ann = annual_vol[day_idx]
        intraday_variance = sigma_ann ** 2 / 252.0
        intraday_sigma_step = np.sqrt(intraday_variance / config.intraday_steps)

        # Overnight move from previous close to today's open.
        overnight_return = config.overnight_vol * local_rng.standard_normal()
        open_price = previous_close * np.exp(overnight_return)

        # Intraday diffusion with drift.
        drift_step = config.annual_drift / 252.0 / config.intraday_steps
        intraday_log_returns = (
            drift_step
            + intraday_sigma_step * local_rng.standard_normal(config.intraday_steps)
        )

        # Occasional jump inside the day.
        jump_return = 0.0
        jump_occurred = local_rng.uniform() < config.jump_probability

        if jump_occurred:
            jump_idx = int(local_rng.integers(0, config.intraday_steps))
            jump_return = config.jump_scale * local_rng.standard_normal()
            intraday_log_returns[jump_idx] += jump_return

        intraday_log_prices = np.log(open_price) + np.cumsum(intraday_log_returns)
        intraday_prices = np.exp(intraday_log_prices)

        high = float(np.max(intraday_prices))
        low = float(np.min(intraday_prices))
        close = float(intraday_prices[-1])

        # The diffusion component's intraday integrated variance is known.
        true_intraday_variance = intraday_variance

        # Jump-inclusive intraday quadratic variation includes jump^2.
        true_jump_inclusive_intraday_qv = intraday_variance + jump_return ** 2

        close_to_close_return = np.log(close) - np.log(previous_close)
        open_to_close_return = np.log(close) - np.log(open_price)

        rows.append({
            "timestamp": date,
            "open": float(open_price),
            "high": high,
            "low": low,
            "close": close,
            "previous_close": float(previous_close),
            "annual_vol_latent": float(sigma_ann),
            "true_intraday_variance": float(true_intraday_variance),
            "true_jump_inclusive_intraday_qv": float(true_jump_inclusive_intraday_qv),
            "overnight_log_return": float(overnight_return),
            "open_to_close_log_return": float(open_to_close_return),
            "close_to_close_log_return": float(close_to_close_return),
            "jump_occurred": bool(jump_occurred),
            "jump_return": float(jump_return),
        })

        previous_close = close

    return pd.DataFrame(rows)


ohlc = simulate_ohlc_panel(config)

ohlc.head()

## 6. Visualising synthetic OHLC data

The simulation creates a realistic-looking price series with time-varying volatility.

The latent annualised volatility is known, which makes this useful for evaluating estimator bias and error.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ohlc["timestamp"], ohlc["close"])
plt.title("Synthetic Close Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ohlc["timestamp"], ohlc["annual_vol_latent"])
plt.title("Latent Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.show()

## 7. Implementing volatility estimators

Each estimator returns a **daily variance estimate**, not annualised volatility.

Annualised volatility is then:

$$
\begin{aligned}
\hat{\sigma}_{\text{ann},t} &= \sqrt{252 \hat{\sigma}_t^2}
\end{aligned}
$$
We clip tiny negative values caused by numerical round-off, but the formulas should generally be non-negative when OHLC data is valid.

In [ ]:
def add_ohlc_volatility_estimators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add close-to-close, open-to-close, Parkinson, Garman-Klass,
    and Rogers-Satchell variance estimators.
    """
    out = df.copy()

    log_o = np.log(out["open"])
    log_h = np.log(out["high"])
    log_l = np.log(out["low"])
    log_c = np.log(out["close"])
    log_prev_c = np.log(out["previous_close"])

    hl = log_h - log_l
    co = log_c - log_o

    out["var_close_to_close"] = (log_c - log_prev_c) ** 2
    out["var_open_to_close"] = co ** 2

    out["var_parkinson"] = (hl ** 2) / (4 * np.log(2))

    out["var_garman_klass"] = (
        0.5 * hl ** 2
        - (2 * np.log(2) - 1) * co ** 2
    )

    out["var_rogers_satchell"] = (
        (log_h - log_o) * (log_h - log_c)
        + (log_l - log_o) * (log_l - log_c)
    )

    estimator_cols = [
        "var_close_to_close",
        "var_open_to_close",
        "var_parkinson",
        "var_garman_klass",
        "var_rogers_satchell",
    ]

    for col in estimator_cols:
        out[col] = out[col].clip(lower=0.0)
        out[f"ann_vol_{col.replace('var_', '')}"] = np.sqrt(252 * out[col])

    return out


vol_panel = add_ohlc_volatility_estimators(ohlc)

vol_panel.head()

## 8. Estimator comparison against known variance

We compare the estimators to two different targets:

### 8.1 Intraday diffusion variance

$$
\sigma^2_{\text{intraday},t}
$$
This excludes overnight gaps and jumps.

Range-based estimators are mainly designed for this kind of intraday continuous-path target.

### 8.2 Jump-inclusive intraday quadratic variation

$$
QV_t = \sigma^2_{\text{intraday},t} + J_t^2
$$
This includes intraday jumps.

Close-to-close variance includes both overnight and intraday movements, so it does not estimate the exact same target as the range-based estimators.

In [ ]:
ESTIMATOR_COLUMNS = [
    "var_close_to_close",
    "var_open_to_close",
    "var_parkinson",
    "var_garman_klass",
    "var_rogers_satchell",
]


def estimator_error_table(
    panel: pd.DataFrame,
    target_col: str,
    estimator_cols: list[str]
) -> pd.DataFrame:
    """
    Compute bias, RMSE, MAE, and correlation against a known target.
    """
    rows = []

    target = panel[target_col].to_numpy()

    for col in estimator_cols:
        estimate = panel[col].to_numpy()
        error = estimate - target

        rows.append({
            "estimator": col.replace("var_", ""),
            "target": target_col,
            "mean_estimate": float(np.mean(estimate)),
            "mean_target": float(np.mean(target)),
            "bias": float(np.mean(error)),
            "relative_bias": float(np.mean(error) / np.mean(target)),
            "rmse": float(np.sqrt(np.mean(error ** 2))),
            "mae": float(np.mean(np.abs(error))),
            "correlation_with_target": float(np.corrcoef(estimate, target)[0, 1]),
        })

    return pd.DataFrame(rows).sort_values("rmse")


intraday_error_table = estimator_error_table(
    vol_panel,
    target_col="true_intraday_variance",
    estimator_cols=ESTIMATOR_COLUMNS
)

jump_inclusive_error_table = estimator_error_table(
    vol_panel,
    target_col="true_jump_inclusive_intraday_qv",
    estimator_cols=ESTIMATOR_COLUMNS
)

intraday_error_table

In [ ]:
jump_inclusive_error_table

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = intraday_error_table.sort_values("rmse", ascending=True)
plt.barh(plot_df["estimator"], plot_df["rmse"])
plt.title("Estimator RMSE vs True Intraday Diffusion Variance")
plt.xlabel("RMSE")
plt.ylabel("Estimator")
plt.show()

## 9. Behaviour on jump days versus non-jump days

Range-based estimators assume a continuous intraday path.

Jumps can distort the range, close-to-close return, and OHLC relationships.

We compare estimator behaviour on jump days and non-jump days.

In [ ]:
jump_summary = (
    vol_panel
    .groupby("jump_occurred")[ESTIMATOR_COLUMNS + ["true_intraday_variance", "true_jump_inclusive_intraday_qv"]]
    .mean()
    .reset_index()
)

jump_summary

In [ ]:
jump_plot = jump_summary.melt(
    id_vars="jump_occurred",
    value_vars=ESTIMATOR_COLUMNS,
    var_name="estimator",
    value_name="mean_variance_estimate"
)

jump_plot["estimator"] = jump_plot["estimator"].str.replace("var_", "", regex=False)

plt.figure(figsize=(11, 6))

for jump_state, group in jump_plot.groupby("jump_occurred"):
    label = "Jump days" if jump_state else "Non-jump days"
    plt.plot(group["estimator"], group["mean_variance_estimate"], marker="o", label=label)

plt.xticks(rotation=30, ha="right")
plt.title("Mean Variance Estimate on Jump vs Non-Jump Days")
plt.xlabel("Estimator")
plt.ylabel("Mean daily variance estimate")
plt.legend()
plt.show()

## 10. Overnight gaps and estimator target mismatch

Close-to-close returns include overnight gaps:

$$
\begin{aligned}
\log C_t - \log C_{t-1} &= (\log O_t - \log C_{t-1}) \\
&\quad + (\log C_t - \log O_t)
\end{aligned}
$$
Range-based estimators use today's open, high, low, and close, so they mostly estimate intraday variation.

This means close-to-close and range-based estimators are not always directly comparable.

If overnight gaps matter for the risk model, they should be measured separately and then combined explicitly.

In [ ]:
vol_panel["var_overnight"] = vol_panel["overnight_log_return"] ** 2
vol_panel["var_intraday_oc"] = vol_panel["open_to_close_log_return"] ** 2
vol_panel["var_overnight_plus_intraday_oc"] = vol_panel["var_overnight"] + vol_panel["var_intraday_oc"]

overnight_summary = pd.Series({
    "mean_close_to_close_variance": vol_panel["var_close_to_close"].mean(),
    "mean_overnight_variance": vol_panel["var_overnight"].mean(),
    "mean_open_to_close_variance": vol_panel["var_intraday_oc"].mean(),
    "mean_overnight_plus_intraday_oc": vol_panel["var_overnight_plus_intraday_oc"].mean(),
    "overnight_share_of_sum": (
        vol_panel["var_overnight"].mean()
        / vol_panel["var_overnight_plus_intraday_oc"].mean()
    )
})

overnight_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(vol_panel["overnight_log_return"], bins=80, alpha=0.7, label="Overnight log return")
plt.hist(vol_panel["open_to_close_log_return"], bins=80, alpha=0.5, label="Open-to-close log return")
plt.title("Overnight vs Open-to-Close Log Returns")
plt.xlabel("Log return")
plt.ylabel("Count")
plt.legend()
plt.show()

## 11. Rolling realised volatility features

Daily variance estimates are noisy.

For forecasting, it is common to build rolling realised volatility features.

For estimator $e$, define:

$$
RV^{(e)}_{t,d} = \hat{\sigma}_{e,t}^2
$$
$$
RV^{(e)}_{t,w} = \frac{1}{5}\sum_{i=0}^{4} RV^{(e)}_{t-i}
$$
$$
RV^{(e)}_{t,m} = \frac{1}{22}\sum_{i=0}^{21} RV^{(e)}_{t-i}
$$
These daily, weekly, and monthly features are the basic inputs to the HAR-RV model.

In [ ]:
def add_har_rv_features(
    panel: pd.DataFrame,
    estimator_col: str,
    weekly_window: int = 5,
    monthly_window: int = 22
) -> pd.DataFrame:
    """
    Add HAR-style daily, weekly, and monthly realised variance features.
    """
    out = panel.copy()

    suffix = estimator_col.replace("var_", "")

    out[f"rv_{suffix}_daily"] = out[estimator_col]
    out[f"rv_{suffix}_weekly"] = out[estimator_col].rolling(weekly_window).mean()
    out[f"rv_{suffix}_monthly"] = out[estimator_col].rolling(monthly_window).mean()

    out[f"ann_vol_{suffix}_weekly"] = np.sqrt(252 * out[f"rv_{suffix}_weekly"])
    out[f"ann_vol_{suffix}_monthly"] = np.sqrt(252 * out[f"rv_{suffix}_monthly"])

    return out


rv_feature_panel = vol_panel.copy()

for estimator_col in ESTIMATOR_COLUMNS:
    rv_feature_panel = add_har_rv_features(rv_feature_panel, estimator_col)

rv_feature_panel.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rv_feature_panel["timestamp"], rv_feature_panel["ann_vol_garman_klass_weekly"], label="GK weekly annualised vol")
plt.plot(rv_feature_panel["timestamp"], rv_feature_panel["ann_vol_rogers_satchell_weekly"], label="RS weekly annualised vol")
plt.plot(rv_feature_panel["timestamp"], rv_feature_panel["annual_vol_latent"], label="Latent annualised vol", alpha=0.8)
plt.title("Rolling Range-Based Volatility vs Latent Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 12. Preparing a HAR-RV target

A simple HAR-RV target is next-day realised variance:

$$
RV_{t+1}
$$
For example, using Rogers-Satchell:

$$
RV^{RS}_{t+1}
$$
The corresponding predictors are:

$$
RV^{RS}_{t,d},\quad RV^{RS}_{t,w},\quad RV^{RS}_{t,m}
$$
This notebook does not fit the HAR-RV model yet. It creates the cleaned feature table for the later forecasting notebook.

In [ ]:
def build_har_ready_table(
    panel: pd.DataFrame,
    estimator: str = "rogers_satchell"
) -> pd.DataFrame:
    """
    Build a HAR-RV-ready table for one estimator.
    """
    daily_col = f"rv_{estimator}_daily"
    weekly_col = f"rv_{estimator}_weekly"
    monthly_col = f"rv_{estimator}_monthly"

    out = panel[[
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "annual_vol_latent",
        "jump_occurred",
        "overnight_log_return",
        daily_col,
        weekly_col,
        monthly_col
    ]].copy()

    out = out.rename(columns={
        daily_col: "rv_daily",
        weekly_col: "rv_weekly",
        monthly_col: "rv_monthly"
    })

    out["rv_next_day_target"] = out["rv_daily"].shift(-1)
    out["log_rv_daily"] = np.log(out["rv_daily"].clip(lower=1e-12))
    out["log_rv_weekly"] = np.log(out["rv_weekly"].clip(lower=1e-12))
    out["log_rv_monthly"] = np.log(out["rv_monthly"].clip(lower=1e-12))
    out["log_rv_next_day_target"] = np.log(out["rv_next_day_target"].clip(lower=1e-12))

    out = out.dropna().reset_index(drop=True)

    out["estimator"] = estimator

    return out


har_rs_table = build_har_ready_table(rv_feature_panel, estimator="rogers_satchell")
har_gk_table = build_har_ready_table(rv_feature_panel, estimator="garman_klass")
har_pk_table = build_har_ready_table(rv_feature_panel, estimator="parkinson")

har_rs_table.head()

## 13. Simple persistence diagnostic

Volatility is persistent.

A quick pre-HAR diagnostic is to check whether today's volatility features correlate with tomorrow's realised variance.

This is not a full forecasting evaluation, but it verifies that the feature table has sensible time-series structure.

In [ ]:
def har_feature_correlation_summary(har_table: pd.DataFrame) -> pd.DataFrame:
    """
    Correlate HAR-style features with the next-day realised variance target.
    """
    feature_cols = [
        "rv_daily",
        "rv_weekly",
        "rv_monthly",
        "log_rv_daily",
        "log_rv_weekly",
        "log_rv_monthly"
    ]

    rows = []

    for col in feature_cols:
        rows.append({
            "feature": col,
            "correlation_with_next_day_rv": har_table[col].corr(har_table["rv_next_day_target"]),
            "correlation_with_next_day_log_rv": har_table[col].corr(har_table["log_rv_next_day_target"])
        })

    return pd.DataFrame(rows)


har_correlation_summary = har_feature_correlation_summary(har_rs_table)

har_correlation_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(
    har_correlation_summary["feature"],
    har_correlation_summary["correlation_with_next_day_log_rv"]
)
plt.axvline(0, linewidth=1)
plt.title("HAR Features vs Next-Day Log Realized Variance")
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.show()

## 14. Estimator stability under rolling windows

We compare rolling annualised volatility estimates across estimators.

This helps detect whether one estimator is unusually unstable, jump-sensitive, or smooth.

In [ ]:
rolling_comparison = rv_feature_panel[[
    "timestamp",
    "annual_vol_latent",
    "ann_vol_close_to_close_weekly",
    "ann_vol_open_to_close_weekly",
    "ann_vol_parkinson_weekly",
    "ann_vol_garman_klass_weekly",
    "ann_vol_rogers_satchell_weekly"
]].copy()

rolling_comparison = rolling_comparison.dropna()

rolling_comparison.head()

In [ ]:
plt.figure(figsize=(12, 6))

for col in [
    "ann_vol_close_to_close_weekly",
    "ann_vol_parkinson_weekly",
    "ann_vol_garman_klass_weekly",
    "ann_vol_rogers_satchell_weekly"
]:
    plt.plot(rolling_comparison["timestamp"], rolling_comparison[col], label=col.replace("ann_vol_", ""))

plt.plot(
    rolling_comparison["timestamp"],
    rolling_comparison["annual_vol_latent"],
    label="latent_vol",
    linewidth=2,
    alpha=0.85
)

plt.title("Weekly Annualised Volatility Estimates")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 15. Estimator diagnostics summary

We create a compact report combining:

- bias;
- RMSE;
- MAE;
- correlation with latent variance;
- jump-day sensitivity;
- overnight mismatch.

This report can be saved and used to justify which estimator feeds later forecasting models.

In [ ]:
def build_estimator_diagnostics_report(
    panel: pd.DataFrame,
    intraday_error_table: pd.DataFrame,
    jump_inclusive_error_table: pd.DataFrame
) -> pd.DataFrame:
    """
    Build a compact estimator diagnostics report.
    """
    jump_means = (
        panel
        .groupby("jump_occurred")[ESTIMATOR_COLUMNS]
        .mean()
        .T
        .reset_index()
        .rename(columns={"index": "estimator", False: "mean_non_jump", True: "mean_jump"})
    )

    jump_means["estimator"] = jump_means["estimator"].str.replace("var_", "", regex=False)
    jump_means["jump_to_non_jump_ratio"] = jump_means["mean_jump"] / jump_means["mean_non_jump"]

    report = intraday_error_table.merge(
        jump_inclusive_error_table[[
            "estimator",
            "rmse",
            "relative_bias"
        ]].rename(columns={
            "rmse": "rmse_vs_jump_inclusive_qv",
            "relative_bias": "relative_bias_vs_jump_inclusive_qv"
        }),
        on="estimator",
        how="left"
    )

    report = report.merge(
        jump_means[["estimator", "jump_to_non_jump_ratio"]],
        on="estimator",
        how="left"
    )

    return report.sort_values("rmse")


estimator_diagnostics_report = build_estimator_diagnostics_report(
    vol_panel,
    intraday_error_table,
    jump_inclusive_error_table
)

estimator_diagnostics_report

## 16. Saving realised-volatility feature tables

The notebook saves:

1. full OHLC panel with all estimators;
2. estimator diagnostics report;
3. HAR-ready Rogers-Satchell table;
4. HAR-ready Garman-Klass table;
5. HAR-ready Parkinson table;
6. simulation manifest.

These outputs can be used by the later HAR-RV notebook.

In [ ]:
output_dir = Path("data/processed/realized_volatility_estimators_v1")
output_dir.mkdir(parents=True, exist_ok=True)

vol_panel_path = output_dir / "ohlc_with_realized_vol_estimators.csv"
diagnostics_path = output_dir / "estimator_diagnostics_report.csv"
har_rs_path = output_dir / "har_rv_ready_rogers_satchell.csv"
har_gk_path = output_dir / "har_rv_ready_garman_klass.csv"
har_pk_path = output_dir / "har_rv_ready_parkinson.csv"
manifest_path = output_dir / "manifest.json"

rv_feature_panel.to_csv(vol_panel_path, index=False)
estimator_diagnostics_report.to_csv(diagnostics_path, index=False)
har_rs_table.to_csv(har_rs_path, index=False)
har_gk_table.to_csv(har_gk_path, index=False)
har_pk_table.to_csv(har_pk_path, index=False)

manifest = {
    "dataset_name": "synthetic_realized_volatility_estimators",
    "schema_version": "realized_volatility_estimators_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "estimators": [
        "close_to_close",
        "open_to_close",
        "parkinson",
        "garman_klass",
        "rogers_satchell"
    ],
    "notes": [
        "Range-based estimators mainly target intraday/open-to-close variance.",
        "Close-to-close includes overnight moves.",
        "Simulation includes time-varying volatility, overnight gaps, drift, and jumps.",
        "HAR-ready tables are for later forecasting notebooks."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

vol_panel_path, diagnostics_path, har_rs_path, manifest_path

## 17. Practical checklist for choosing a realised-volatility estimator

Before choosing an estimator, ask:

1. **What risk horizon matters?**  
   Intraday, close-to-close, overnight-inclusive, or total risk?

2. **Do you have reliable OHLC data?**  
   Bad highs/lows can destroy range-based estimators.

3. **Are jumps common?**  
   Classical range estimators assume continuous paths.

4. **Are overnight gaps important?**  
   Range-based estimators using $O,H,L,C$ generally miss close-to-open gap risk.

5. **Is drift material?**  
   Rogers-Satchell is more robust to drift than some alternatives.

6. **Are price limits present?**  
   Futures with limit-up/limit-down days may have censored highs/lows.

7. **Is the estimator used for forecasting or risk reporting?**  
   HAR-RV forecasting may prefer a stable estimator, while risk reporting may need jump-inclusive measures.

8. **Is the downstream model calibrated to the estimator?**  
   A model trained on Parkinson RV is not necessarily interchangeable with one trained on close-to-close RV.

9. **Are estimates annualised consistently?**  
   Daily variance and annualised volatility are not the same unit.

10. **Do you preserve estimator metadata?**  
   Downstream notebooks should know which estimator produced the feature.

## 18. Limitations

This notebook deliberately uses synthetic OHLC data.

### 18.1 Synthetic highs and lows are cleaner than real data

Real OHLC data can contain:

- bad highs/lows;
- auction prints;
- exchange corrections;
- stale bars;
- missing opens;
- price-limit censoring;
- vendor-specific adjustment rules.

Range-based estimators are very sensitive to bad high/low values.

### 18.2 Continuous-path assumptions are violated by jumps

Parkinson, Garman-Klass, and Rogers-Satchell are derived under continuous log-price assumptions.

Real markets have jumps caused by news, macro releases, liquidity shocks, and limit moves.

### 18.3 Overnight variance is separate

Range estimators based on today's open, high, low, and close do not automatically include close-to-open gap risk.

For overnight-sensitive strategies, estimate overnight variance separately.

### 18.4 Drift and microstructure matter

Garman-Klass is efficient under ideal assumptions but can be biased when drift or jumps matter.

Rogers-Satchell is more drift-robust but still assumes reliable OHLC data.

### 18.5 HAR-RV feature construction is only the next step

Creating daily, weekly, and monthly realised-volatility features does not prove forecasting power.

The HAR-RV model must be trained and evaluated out of sample with proper time-series validation.

### 18.6 Futures-specific issues are not fully modelled

For Chinese futures, realised volatility estimates may be affected by:

- night sessions;
- lunch breaks;
- price limits;
- contract rolling;
- margin changes;
- dominant-contract definitions;
- exchange holiday calendars.

These should be handled before using realised volatility in a live-style backtest.

## 19. Research frontier and current directions

Realised volatility estimation remains active because markets are noisy, discontinuous, and high-dimensional.

### 19.1 Noise-robust high-frequency volatility

With tick or intraday data, realised variance can be biased by market microstructure noise.

Research methods include:

- realised kernels;
- pre-averaging;
- subsampling;
- two-scale estimators;
- noise-robust jump tests.

A future notebook could compare OHLC estimators with intraday realised variance and realised kernel estimators.

### 19.2 Jump-robust volatility and bipower variation

Realised variance includes jumps.

If the goal is to separate continuous volatility from jumps, bipower variation and related estimators can be useful.

A future notebook could simulate jumps and decompose total quadratic variation into continuous and jump components.

### 19.3 Yang-Zhang and overnight-aware OHLC estimators

Yang-Zhang-type estimators combine overnight, open-to-close, and range-based components.

They are useful when both intraday and overnight variation matter.

A future notebook could add Yang-Zhang and compare it with the four estimators covered here.

### 19.4 HAR-RV and long-memory volatility forecasting

The HAR-RV model uses daily, weekly, and monthly realised volatility components to capture volatility persistence across heterogeneous horizons.

A future notebook will use the HAR-ready tables from this notebook to forecast realised volatility.

### 19.5 Realised volatility with rough volatility models

Modern volatility research often treats volatility as rough rather than smooth.

A future notebook could compare HAR-RV features with rough-volatility-inspired features.

### 19.6 Options-implied and realised volatility hybrids

A modern forecasting question is whether options-implied measures improve forecasts beyond historical realised volatility.

A future notebook could augment HAR-RV with implied volatility, volatility risk premium, or model-based spot volatility.

## 20. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_02_har_realized_volatility_forecasting**  
   Forecasting realised volatility using daily, weekly, and monthly components.

2. **03_01_garch_volatility_modeling**  
   Comparing return-based GARCH volatility with realised-volatility features.

3. **04_06_var_cvar_and_stress_testing**  
   Using realised volatility estimates for risk scaling and stress scenarios.

4. **05_01_vectorized_backtest_engine**  
   Applying volatility targeting using realised volatility estimates.

5. **06_05_rough_volatility_estimation**  
   Studying volatility roughness and memory.

6. **02_11_heston_model_calibration**  
   Connecting realised volatility diagnostics to stochastic volatility models.

7. **07_01_multi_asset_cta_research_pipeline**  
   Using realised volatility for risk targeting in an end-to-end CTA system.

## 21. Summary

This notebook compared several OHLC realised-volatility estimators:

1. close-to-close;
2. open-to-close;
3. Parkinson;
4. Garman-Klass;
5. Rogers-Satchell.

It showed that:

- close-to-close includes overnight gaps;
- range-based estimators use more intraday information;
- Parkinson is simple and efficient under ideal conditions;
- Garman-Klass uses OHLC information but assumes zero drift/no jumps;
- Rogers-Satchell is more robust to drift;
- jumps and overnight gaps can change estimator interpretation;
- HAR-RV forecasting requires a consistent realised-volatility input.

The key computational takeaway is:

> A realised-volatility estimator should be stored with its assumptions, target, unit, and transformation. Otherwise downstream models may mix incompatible volatility definitions.

The key financial takeaway is:

> The best volatility estimator depends on the market, horizon, data quality, and use case. There is no universal volatility number.

## 22. Further reading

### Classical OHLC estimators

- Parkinson, M. "The Extreme Value Method for Estimating the Variance of the Rate of Return."
- Garman, M. B. and Klass, M. J. "On the Estimation of Security Price Volatilities from Historical Data."
- Rogers, L. C. G. and Satchell, S. E. "Estimating Variance from High, Low and Closing Prices."
- Yang, D. and Zhang, Q. "Drift-Independent Volatility Estimation Based on High, Low, Open, and Close Prices."

### Realised volatility and forecasting

- Andersen, T. G. and Bollerslev, T. work on realised volatility.
- Corsi, F. "A Simple Approximate Long-Memory Model of Realized Volatility."
- Barndorff-Nielsen and Shephard on realised variation and bipower variation.

### Future extensions

- Intraday realised variance.
- Realised kernels.
- Bipower variation.
- Jump detection.
- HAR-RV forecasting.
- Realised volatility versus implied volatility.
- Rough volatility features.